# AgriSmart AI - Model 2 Self-Contained Orchestration Notebook (v2)
=========================================================================

**Field-Domain Adaptation Training Pipeline for EfficientNet-B2**

This notebook is completely self-contained for Google Colab T4 GPU runtimes.
It orchestrates the repository's single source of truth scripts.

### Quick Start:
1. Connect to a **GPU Runtime** (Runtime -> Change runtime type -> T4 GPU).
2. Click **Runtime -> Run all**.

---

### CELL 1 - ENVIRONMENT & GPU DETECTION
Detects CUDA GPU, installs required dependencies, requires GPU, and sets global random seed.

In [ ]:
# Cell 1: Environment, GPU Detection & Random Seeds
import os
import sys
import subprocess
import random
import numpy as np
import torch

# Safe UTF-8 output
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

print("[CELL 1] Environment setup & GPU check...")

# Install required packages quietly
reqs = ["timm", "huggingface_hub", "albumentations", "scikit-learn", "tqdm", "pillow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)

# Detect CUDA
if not torch.cuda.is_available():
    raise RuntimeError("[CRITICAL ERROR] CUDA is not available. Switch runtime type to T4 GPU!")

gpu_name = torch.cuda.get_device_name(0)
print(f"[OK] CUDA GPU Detected: {gpu_name}")
print(f"     PyTorch Version:  {torch.__version__}")

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"[OK] Global random seed set to {SEED}.")

### CELL 2 - CLONE REPOSITORY & WORKSPACE SETUP
Clones AgriSmart-AI repository into `/content/AgriSmart-AI` and sets single `REPO_ROOT` variable.

In [ ]:
# Cell 2: Clone Repository & Workspace Setup
from pathlib import Path

repo_url = "https://github.com/Parrthiv125/AgriSmart-AI.git"
colab_target = Path("/content/AgriSmart-AI")

if colab_target.exists():
    REPO_ROOT = colab_target
    os.chdir(REPO_ROOT)
    subprocess.run(["git", "checkout", "main"], check=True)
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    if Path("models/classes.json").exists():
        REPO_ROOT = Path(os.getcwd()).resolve()
    else:
        print(f"Cloning {repo_url} into {colab_target}...")
        subprocess.run(["git", "clone", repo_url, str(colab_target)], check=True)
        REPO_ROOT = colab_target
        os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"[OK] Repository Root (REPO_ROOT): {REPO_ROOT}")
print(f"[OK] Current Commit Hash:          {commit_hash}")

### CELL 3 - MODEL 1 CHECKPOINT INTEGRITY
Verifies Model 1 checkpoint SHA256 (`models/agrismart_best.pth`). Model 1 MUST remain untouched.

In [ ]:
# Cell 3: Model 1 Checkpoint Verification
from data.common import MODEL1_CKPT_PATH, EXPECTED_MODEL1_HASH, compute_file_sha256

if not MODEL1_CKPT_PATH.exists():
    raise FileNotFoundError(f"[CRITICAL ERROR] Model 1 checkpoint missing at {MODEL1_CKPT_PATH}!")

current_hash = compute_file_sha256(MODEL1_CKPT_PATH)
if current_hash != EXPECTED_MODEL1_HASH:
    raise ValueError(f"[CRITICAL ERROR] Model 1 SHA256 mismatch! Expected {EXPECTED_MODEL1_HASH}, got {current_hash}")

print(f"[OK] Model 1 SHA256 verified: {current_hash[:16]}... (UNTOUCHED)")

ckpt = torch.load(MODEL1_CKPT_PATH, map_location="cpu", weights_only=False)
print(f"[OK] Model 1 Architecture: {ckpt.get('model_name', 'efficientnet_b2')}")
print(f"[OK] Model 1 Classes:      {ckpt.get('num_classes', 28)}")
print(f"[OK] Model 1 Input Res:    {ckpt.get('image_size', 260)}x{ckpt.get('image_size', 260)}")
print(f"[OK] Model 1 Best Epoch:   {ckpt.get('epoch')}")
print(f"[OK] Model 1 Val F1:       {ckpt.get('val_macro_f1', 0.0):.4f}")

### CELL 4 - DOWNLOAD PLANTVILLAGE DATASET
Downloads PlantVillage dataset using `data.download_dataset.download_and_extract_plantvillage()`.

In [ ]:
# Cell 4: Idempotent PlantVillage Download & Fast Extraction
from data.download_dataset import download_and_extract_plantvillage

print("[CELL 4] Downloading & preparing PlantVillage dataset...")
raw_color_dir = download_and_extract_plantvillage()
print(f"[OK] PlantVillage RGB data ready at: {raw_color_dir}")

### CELL 5 - PLANTVILLAGE 28-CLASS SPLIT
Executes PlantVillage 80/10/10 split using `data.split_dataset.split_dataset()`.

In [ ]:
# Cell 5: Execute PlantVillage 28-Class Split
from data.split_dataset import split_dataset

print("[CELL 5] Executing PlantVillage 28-class split (80/10/10, seed 42)...")
stats = split_dataset()
print(f"[OK] PlantVillage Split Complete: Train={stats['splits']['train']['total']} | Val={stats['splits']['val']['total']} | Test={stats['splits']['test']['total']}")

### CELL 6 - DOWNLOAD & PREPARE PLANTDOC DATASET
Downloads PlantDoc dataset using `data.download_plantdoc` and `data.prepare_plantdoc`.

In [ ]:
# Cell 6: Download & Prepare PlantDoc Dataset
from data.download_plantdoc import download_plantdoc
from data.prepare_plantdoc import prepare_plantdoc_dataset

print("[CELL 6] Downloading PlantDoc raw dataset...")
download_plantdoc()

print("Organizing PlantDoc train & test sets...")
pd_stats = prepare_plantdoc_dataset()
print(f"[OK] PlantDoc Ready: Train={pd_stats['counts']['train']['total']} | LOCKED Test={pd_stats['counts']['test']['total']}")

### CELL 7 - COMBINED MODEL 2 DATASET PREPARATION & DEDUPLICATION
Executes combined Model 2 dataset preparation using `data.prepare_model2_dataset.prepare_model2_dataset()`.

In [ ]:
# Cell 7: Combined Model 2 Dataset Preparation with SHA256 Deduplication
from data.prepare_model2_dataset import prepare_model2_dataset

print("[CELL 7] Building combined Model 2 dataset with SHA256 content deduplication...")
model2_stats = prepare_model2_dataset()
print(f"[OK] Dataset Preparation Complete!")
print(f"     Excluded True Content Duplicates: {model2_stats['excluded_duplicates_count']}")

### CELL 8 - AUTHORITATIVE SHA256 CONTENT LEAKAGE AUDIT
Executes single authoritative leakage audit via `data.leakage_checker.audit_content_leakage()`.

In [ ]:
# Cell 8: Authoritative SHA256 Leakage Audit
from data.leakage_checker import audit_content_leakage

print("[CELL 8] Running single authoritative SHA256 content leakage audit...")
audit = audit_content_leakage(verbose=True)
if not audit["is_clean"]:
    raise RuntimeError("[CRITICAL ERROR] Content leakage detected! Re-run cell 7 or inspect dataset.")

### CELL 9 - FIELD DATA SAMPLING CONFIGURATION
Configures `WeightedRandomSampler` with `plantdoc_oversample_factor = 5.0`.

In [ ]:
# Cell 9: Field Data Sampling Strategy
plantdoc_oversample_factor = 5.0

print(f"[OK] Configured PlantDoc Oversample Factor: {plantdoc_oversample_factor}x")
print("     Method: PyTorch WeightedRandomSampler (Zero physical duplication)")
print("     Weighting: PlantVillage samples (w=1.0) | PlantDoc samples (w=5.0)")

### CELL 10 - MODEL 2 PIPELINE & GUARDRAIL VERIFICATION
Verifies all pre-training guardrails using `data.verify_model2_pipeline.verify_model2_pipeline()`.

In [ ]:
# Cell 10: Model 2 Pipeline & Guardrail Verification
from data.verify_model2_pipeline import verify_model2_pipeline

print("[CELL 10] Running pre-training verification script...")
success = verify_model2_pipeline()
if not success:
    raise RuntimeError("[CRITICAL ERROR] Pipeline verification failed! Stopping notebook before training.")
print("[OK] All pre-training verifications & guardrails passed!")

### CELL 11 - DRY RUN TRAINING TEST
Executes a fast dry-run test using `training.train_model2.run_training_model2(is_dry_run=True)`.

In [ ]:
# Cell 11: Execute Pipeline Dry-Run
from training.train_model2 import run_training_model2

print("[CELL 11] Executing pipeline dry-run...")
dry_run_stats = run_training_model2(is_dry_run=True, plantdoc_oversample_factor=plantdoc_oversample_factor)
print("[OK] Dry-run completed cleanly.")

### CELL 12 & 13 - MODEL 2 FULL TRAINING & AUTOMATIC RESUME SUPPORT
Trains Model 2 for 25 epochs. Automatically resumes if `models/agrismart_field_adapted_last.pth` exists.

In [ ]:
# Cell 12 & 13: Full Model 2 Training & Automatic Resume Support
print("[CELL 12 & 13] Launching Model 2 Full Field-Domain Adaptation Training (25 Epochs)...")
print("        - Model 1 Checkpoint: models/agrismart_best.pth (UNTOUCHED)")
print("        - PlantDoc TEST Set:  data/plantdoc/test (LOCKED & UNTOUCHED)")
print(f"        - PlantDoc Oversample Factor: {plantdoc_oversample_factor}x")
print()

final_metadata = run_training_model2(
    epochs=25,
    batch_size=32,
    lr=1e-4,
    plantdoc_oversample_factor=plantdoc_oversample_factor,
    is_dry_run=False,
)

### CELL 14 - FINAL SUMMARY & GUARDRAIL AUDIT
Displays final training metrics and confirms that locked test sets remained untouched.

In [ ]:
# Cell 14: Final Training Summary & Guardrail Audit
from data.common import (
    MODEL1_CKPT_PATH,
    EXPECTED_MODEL1_HASH,
    MODEL2_CKPT_OUT,
    MODEL2_LAST_OUT,
    MODEL2_METADATA_OUT,
    PD_TEST_DIR,
    PV_TEST_DIR,
    compute_file_sha256,
)

print("=" * 70)
print(" FINAL MODEL 2 TRAINING SUMMARY & GUARDRAIL AUDIT")
print("=" * 70)

if MODEL2_METADATA_OUT.exists():
    with open(MODEL2_METADATA_OUT, "r", encoding="utf-8") as f:
        meta = json.load(f)
    print(f"  Best Epoch:                  {meta['training_results']['best_epoch']}")
    print(f"  Best Validation Macro-F1:    {meta['training_results']['best_val_macro_f1']:.4f}")
    print(f"  Sampling Method:             {meta['sampling_strategy']['method']}")
    print(f"  PlantDoc Oversample Factor:  {meta['sampling_strategy']['plantdoc_oversample_factor']}x")

print(f"  Best Checkpoint Saved:       {MODEL2_CKPT_OUT} ({MODEL2_CKPT_OUT.stat().st_size / 1e6:.1f} MB)")
print(f"  Last Checkpoint Saved:       {MODEL2_LAST_OUT} ({MODEL2_LAST_OUT.stat().st_size / 1e6:.1f} MB)")

# Verify Model 1 SHA256 integrity
m1_hash = compute_file_sha256(MODEL1_CKPT_PATH)
print(f"  Model 1 SHA256 Integrity:    {'VERIFIED UNTOUCHED' if m1_hash == EXPECTED_MODEL1_HASH else 'FAILED Altered!'}")

# Confirm Locked Test Sets
pd_test_count = len(list(PD_TEST_DIR.rglob("*"))) if PD_TEST_DIR.exists() else 0
pv_test_count = len(list(PV_TEST_DIR.rglob("*"))) if PV_TEST_DIR.exists() else 0
print(f"  PlantDoc TEST Status:        LOCKED & UNTOUCHED ({pd_test_count} items)")
print(f"  PlantVillage TEST Status:     LOCKED & UNTOUCHED ({pv_test_count} items)")
print(f"  PlantDoc TEST Evaluated:     FALSE (Evaluation locked until model frozen)")
print("=" * 70)
print(" [SUCCESS] MODEL 2 TRAINING PIPELINE FINISHED SUCCESSFULLY.")